In [ ]:
FIGURE = 'Fig.S5'
from pathlib import Path
from contextlib import contextmanager
import os
import sys
import gc
import mmap
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
import netCDF4 as nc
_NETCDF_DATASET = nc.Dataset
HERE = Path.cwd().resolve()
if not (HERE / 'Inputs').is_dir():
    for candidate in [HERE / 'Figure' / FIGURE, HERE / FIGURE, HERE / 'Data Processing' if FIGURE in ['SSMIrz','Rmax_Ts'] else HERE]:
        if (candidate / 'Inputs').is_dir():
            HERE = candidate
            break
INPUT_DIR = HERE / 'Inputs'
OUTPUT_DIR = HERE / 'Outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
ROOT = HERE
DATA = INPUT_DIR
OUTPUT = OUTPUT_DIR
__file__ = str(HERE / (FIGURE + '.ipynb'))
sys.argv = ['notebook']

def open_nc(path, mode='r', **kwargs):
    if 'memory' in kwargs:
        return _NETCDF_DATASET(path, mode, **kwargs)
    path = Path(path).resolve()
    previous = Path.cwd()
    try:
        os.chdir(path.parent)
        return _NETCDF_DATASET(path.name, mode, **kwargs)
    finally:
        os.chdir(previous)

def asset_path(key):
    if key in ['asset_e2603c619a67', 'asset_5e3c32038bfb']:
        return INPUT_DIR / 'study_mask.tif'
    if key == 'asset_beb53c9b3525':
        return INPUT_DIR / 'SPEI_k01-k24.nc'
    if key == 'asset_174bc324d489':
        return INPUT_DIR / 'SSMIrz_k01-k24.nc'
    return OUTPUT_DIR

def output_path(name=''):
    path = OUTPUT_DIR / name
    path.mkdir(parents=True, exist_ok=True)
    return path

def module_path(name=''):
    return str(HERE)

def font_path(filename):
    from matplotlib.font_manager import FontProperties, findfont
    family = 'Times New Roman' if filename.lower().startswith('times') else 'Courier New' if filename.lower().startswith('cour') else 'Arial'
    return findfont(FontProperties(family=family))

def array(value):
    return np.asarray(np.ma.filled(value.astype(float), np.nan))

@contextmanager
def mapped(path):
    with Path(path).open('rb') as stream:
        mm = mmap.mmap(stream.fileno(), 0, access=mmap.ACCESS_READ)
        dataset = open_nc('input.nc', memory=mm)
        try:
            yield dataset
        finally:
            dataset.close()
            gc.collect()
            mm.close()


In [ ]:
from pathlib import Path
import json
import re
import netCDF4 as nc
import numpy as np
from PIL import Image, ImageDraw, ImageFont
OUT_ROOT = OUTPUT / 'Fig.S4-S6'
TMP_COMPAT_DIR = OUTPUT / 'runtime/sensitivity_maps'
METRICS = {v: {p: INPUT_DIR / f'{v}_{p}.nc' for p in ['SPEI', 'SSMIrz']} for v in ['LAI']}
SEASONS = [('MAM', 'Spring', 0)]
PATHWAYS = ['SPEI', 'SSMIrz']
FONT_ARIAL = font_path('arial.ttf')
FONT_TIMES = font_path('times.ttf')
CROP_TOP = 220
ROW_GAP = 2
COL_GAP = 16
LETTER_SIZE = 243
SEASON_SIZE = 267
TITLE_SIZE = 267
TITLE_SUB_SIZE = 160
TITLE_SUB_YOFF = 107
COLORBAR_OUTLINE_W = 7

def draw_reference_panel(source_nc, target_image):
    import xarray as xr
    import matplotlib.pyplot as plt
    import numpy as np
    import cartopy.crs as ccrs
    from matplotlib.colors import ListedColormap, BoundaryNorm
    import matplotlib as mpl
    import rasterio
    import matplotlib.pyplot as plt
    from mpl_toolkits.axes_grid1 import make_axes_locatable
    import cartopy.feature as cfeature
    mpl.rcParams['font.family'] = 'Times New Roman'
    mpl.rcParams['font.size'] = 12
    mask_path = str(INPUT_DIR / 'study_mask.tif')
    boundaries = [-1, -0.8, -0.6, -0.4, -0.2, 0, 0.2, 0.4, 0.6, 0.8, 1]
    colors = ['#2D88A9', '#45C5F7', '#7ED1EF', '#B3E2FB', '#DAF1FD', '#FCEEC1', '#FBE7A5', '#FAD07A', '#F3A800', '#9E6E12']
    cmap = ListedColormap(colors, 'indexed')
    norm = BoundaryNorm(boundaries, ncolors=len(colors), clip=True)
    for start_year in range(2001, 2002):
        end_year = start_year + 19
        nc_file_name = f'max_corr_{start_year}-{end_year}_Spring.nc'
        nc_file_path = str(source_nc)
        ds = xr.open_dataset(nc_file_path, engine='h5netcdf')
        max_correlation = ds[f'max_correlation'].values
        lons = ds['lon'].values
        lats = ds['lat'].values
        with rasterio.open(mask_path) as src:
            mask = src.read(1)
            mask = np.flipud(mask)
        max_correlation[(mask == 0) | (max_correlation == 0)] = np.nan
        lat_mean_corr = np.nanmean(max_correlation, axis=1)
        lon_mean_corr = np.nanmean(max_correlation, axis=0)
        fig = plt.figure(figsize=(14, 10), dpi=600)
        proj = ccrs.PlateCarree()
        ax = fig.add_subplot(2, 1, 1, projection=proj)
        cmesh = ax.pcolormesh(lons, lats, max_correlation, shading='auto', cmap=cmap, norm=norm, transform=ccrs.PlateCarree())
        ax.set_title(f'Max correlation of {start_year}-{end_year} MAM', fontsize=24)
        ax.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND, facecolor='whitesmoke', edgecolor='none', zorder=0)
        cmesh = ax.contourf(lons, lats, max_correlation, shading='auto', cmap=cmap, norm=norm, transform=ccrs.PlateCarree(), zorder=1)
        ax.add_feature(cfeature.LAND, facecolor='none', edgecolor='black', zorder=4)
        lon_ticks = np.linspace(-180, 180, 7)
        lat_ticks = np.linspace(-90, 90, 7)
        ax.set_xticks(lon_ticks, crs=proj)
        ax.set_yticks(lat_ticks, crs=proj)
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.yaxis.set_ticks_position('right')
        ax.yaxis.set_label_position('right')
        ax.tick_params(axis='x', direction='in')
        ax.tick_params(axis='y', direction='in')
        gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = True
        gl.bottom_labels = True
        gl.left_labels = False
        gl.xlabel_style = {'rotation': 0, 'ha': 'center'}
        gl.ylabel_style = {'rotation': 270, 'va': 'center'}
        colors = ['#2D88A9', '#45C5F7', '#7ED1EF', '#B3E2FB', '#DAF1FD', '#FCEEC1', '#FBE7A5', '#FAD07A', '#F3A800', '#9E6E12']
        boundaries = [-1, -0.8, -0.6, -0.4, -0.2, 0, 0.2, 0.4, 0.6, 0.8, 1]

        def max_balance_method(percentages, total=100, precision=2):
            percentages_decimal = [p / total for p in percentages]
            total_decimal = sum(percentages_decimal)
            deficit = 1 - total_decimal
            adjustments = [p * deficit / total_decimal for p in percentages_decimal]
            adjusted_percentages = [round(p * total + a * total, precision) for p, a in zip(percentages_decimal, adjustments)]
            total_adjusted = sum(adjusted_percentages)
            diff = total - total_adjusted
            if diff != 0:
                adjusted_percentages[-1] += diff
            adjusted_percentages = [max(0, val) for val in adjusted_percentages]
            return adjusted_percentages
        latitudes = ds['lat'].values
        lat_weights = np.cos(np.deg2rad(latitudes))
        lat_weights = np.tile(lat_weights[:, np.newaxis], (1, len(lons)))
        flat_correlation = max_correlation.flatten()
        flat_weights = lat_weights.flatten()
        n_intervals = len(boundaries) - 1
        percentages = np.zeros(n_intervals)
        for i in range(n_intervals):
            lower = boundaries[i]
            upper = boundaries[i + 1]
            mask = (flat_correlation >= lower) & (flat_correlation < upper) & ~np.isnan(flat_correlation)
            total_weight = np.sum(flat_weights[mask])
            percentages[i] = total_weight
        total_weight = np.sum(flat_weights[~np.isnan(flat_correlation)])
        percentages = percentages / total_weight * 100
        percentages = np.round(percentages, 2)
        percentages = max_balance_method(percentages.tolist(), total=100, precision=2)
        labels = [f'{boundaries[i]:.1f}-{boundaries[i + 1]:.1f}' for i in range(n_intervals)]
        ax_hist = fig.add_axes([0.23, 0.55, 0.1, 0.2])
        bars = ax_hist.barh(labels, percentages, color=[colors[i] for i in range(n_intervals)])
        for i, (value, bar) in enumerate(zip(percentages, bars)):
            bar_width = bar.get_width()
            if bar_width >= 15:
                x_pos = bar_width
                ha = 'right'
            else:
                x_pos = bar_width + 1
                ha = 'left'
            ax_hist.text(x_pos, i, f'{value:.2f}%', va='center', ha=ha)
        ax_hist.set_title('Percentage Distribution', fontsize=8)
        ax_hist.set_xlabel('Percentage', fontsize=8)
        ax_hist.set_ylabel('Correlation Range', fontsize=8)
        ax_hist.tick_params(axis='both', which='major', labelsize=8)
        ax_hist.patch.set_alpha(0.5)
        ax_hist.tick_params(axis='y', labelrotation=45)
        ax2 = fig.add_axes([0.85, 0.504, 0.05, 0.447])
        bars = ax2.barh(lats, lat_mean_corr, height=0.5, color=np.where(lat_mean_corr > 0, '#FAD07A', '#45C5F7'))
        ax2.set_yticks(np.linspace(-90, 90, 7))
        ax2.set_yticklabels([])
        ax2.set_ylim(-90, 90)
        ax2.set_xlim(-1, 1)
        ax2.set_xticks(np.linspace(-1, 1, 3))
        ax2.set_xlabel('', fontname='Times New Roman')
        ax2.set_title('', rotation=270)
        ax2.tick_params(axis='x', direction='in')
        ax2.tick_params(axis='y', direction='in', pad=9)
        plt.tight_layout()
        output_filename = f'max_correlation_{start_year}-{end_year}_Spring.jpg'
        output_path = str(target_image)
        plt.savefig(output_path, bbox_inches='tight', pad_inches=0.1, dpi=600)
        plt.close(fig)

def decode_seasons(chars) -> list[str]:
    arr = np.asarray(chars)
    return [b''.join(row).decode('ascii').strip() for row in arr]

def make_compatible_nc(metric: str, pathway: str, source_nc: Path, season_code: str) -> Path:
    with open_nc(source_nc) as src:
        season_names = decode_seasons(src.variables['season_name'][:])
        season_index = season_names.index(season_code)
        lat = np.asarray(src.variables['lat'][:], dtype='float32')
        lon = np.asarray(src.variables['lon'][:], dtype='float32')
        arr = np.ma.filled(src.variables['max_r'][season_index, :, :], np.nan).astype('float32')
    TMP_COMPAT_DIR.mkdir(parents=True, exist_ok=True)
    compat_nc = TMP_COMPAT_DIR / f'compat_max_corr_2001-2020_{metric}_{pathway}_{season_code}.nc'
    if compat_nc.exists():
        compat_nc.unlink()
    with open_nc(compat_nc, 'w', format='NETCDF4') as dst:
        dst.createDimension('lat', len(lat))
        dst.createDimension('lon', len(lon))
        dst.createVariable('lat', 'f4', ('lat',))[:] = lat
        dst.createVariable('lon', 'f4', ('lon',))[:] = lon
        vr = dst.createVariable('max_correlation', 'f4', ('lat', 'lon'), zlib=True, complevel=4, fill_value=np.nan)
        vr[:, :] = arr
        vr.long_name = f'{metric}-{pathway} {season_code} signed Ropt, compatible with original plotting notebook'
    return compat_nc

def postprocess_panel(raw_path: Path, final_path: Path, letter: str, season_code: str) -> None:
    im = Image.open(raw_path).convert('RGB')
    draw = ImageDraw.Draw(im)
    draw.rectangle([0, 0, im.width, 240], fill='white')
    font_letter = ImageFont.truetype(FONT_ARIAL, LETTER_SIZE)
    font_season = ImageFont.truetype(FONT_TIMES, SEASON_SIZE)
    letter_y = 260 if letter in {'a', 'e', 'c', 'g'} else 292
    draw.text((205, letter_y), letter, fill='black', font=font_letter)
    draw.text((205, 880), season_code, fill='black', font=font_season)
    im.save(final_path, dpi=(600, 600), quality=95, subsampling=0)

def load_cropped_panel(panel_dir: Path, metric: str, pathway: str, season: str) -> Image.Image:
    path = panel_dir / f'{metric}_{pathway}_{season}_Ropt_without_title_annotations_600dpi.jpg'
    im = Image.open(path).convert('RGB')
    return im.crop((0, CROP_TOP, im.width, im.height))

def stitch_no_stretch(panel_dir: Path, metric: str) -> Image.Image:
    panels = {(p, s): load_cropped_panel(panel_dir, metric, p, s) for p in PATHWAYS for s, _, _ in SEASONS}
    widths = [im.width for im in panels.values()]
    heights = [im.height for im in panels.values()]
    if len(set(widths)) != 1 or len(set(heights)) != 1:
        raise RuntimeError(f'{metric} panel sizes differ after crop: widths={widths}, heights={heights}')
    w = widths[0]
    h = heights[0]
    canvas = Image.new('RGB', (2 * w + COL_GAP, 4 * h + 3 * ROW_GAP), 'white')
    for r, (season, _, _) in enumerate(SEASONS):
        y = r * (h + ROW_GAP)
        canvas.paste(panels['SPEI', season], (0, y))
        canvas.paste(panels['SSMIrz', season], (w + COL_GAP, y))
    return canvas

def draw_centered_text(draw: ImageDraw.ImageDraw, center_x: int, y: int, text: str, font: ImageFont.FreeTypeFont) -> None:
    bbox = draw.textbbox((0, 0), text, font=font)
    tw = bbox[2] - bbox[0]
    draw.text((center_x - tw / 2, y), text, fill='black', font=font)

def draw_centered_title_with_subscript(draw: ImageDraw.ImageDraw, center_x: int, y: int, parts: list[tuple[str, ImageFont.FreeTypeFont, int]]) -> None:
    widths = []
    for text, font, _ in parts:
        bbox = draw.textbbox((0, 0), text, font=font)
        widths.append(bbox[2] - bbox[0])
    total_w = sum(widths)
    x = center_x - total_w / 2
    for (text, font, yoff), tw in zip(parts, widths):
        draw.text((x, y + yoff), text, fill='black', font=font)
        x += tw

def add_column_titles(img: Image.Image, metric: str) -> Image.Image:
    w, h = img.size
    title_h = 260
    out = Image.new('RGB', (w, h + title_h), 'white')
    out.paste(img, (0, title_h))
    draw = ImageDraw.Draw(out)
    panel_w = (w - COL_GAP) // 2
    main_map_center_x = 2829
    left_center = main_map_center_x
    right_center = panel_w + COL_GAP + main_map_center_x
    title_font = ImageFont.truetype(FONT_TIMES, TITLE_SIZE)
    sub_font = ImageFont.truetype(FONT_TIMES, TITLE_SUB_SIZE)
    title_y = 0
    draw_centered_text(draw, left_center, title_y, f'SPEI-{metric}', title_font)
    draw_centered_title_with_subscript(draw, right_center, title_y, [('SSMI', title_font, 0), ('rz', sub_font, TITLE_SUB_YOFF), (f'-{metric}', title_font, 0)])
    return out

def append_colorbar(img: Image.Image) -> Image.Image:
    w, h = img.size
    colors = ['#2D88A9', '#45C5F7', '#7ED1EF', '#B3E2FB', '#DAF1FD', '#FCEEC1', '#FBE7A5', '#FAD07A', '#F3A800', '#9E6E12']
    tick_labels = ['-1', '-0.8', '-0.6', '-0.4', '-0.2', '0', '0.2', '0.4', '0.6', '0.8', '1']
    extra_h = 600
    out = Image.new('RGB', (w, h + extra_h), 'white')
    out.paste(img, (0, 0))
    draw = ImageDraw.Draw(out)
    bar_x0 = int(round(w * 0.2))
    bar_x1 = int(round(w * 0.8))
    bar_w = bar_x1 - bar_x0
    bar_h = 200
    bar_y0 = h + 95
    bar_y1 = bar_y0 + bar_h
    for i, color in enumerate(colors):
        x0 = int(round(bar_x0 + i * bar_w / len(colors)))
        x1 = int(round(bar_x0 + (i + 1) * bar_w / len(colors)))
        draw.rectangle([x0, bar_y0, x1, bar_y1], fill=color)
    edge_color = 'black'
    outline_w = COLORBAR_OUTLINE_W
    draw.rectangle([bar_x0, bar_y0, bar_x1, bar_y1], outline=edge_color, width=outline_w)
    font_tick = ImageFont.truetype(FONT_TIMES, 225)
    tick_len = 42
    for i, label in enumerate(tick_labels):
        x = int(round(bar_x0 + i * bar_w / (len(tick_labels) - 1)))
        draw.line([x, bar_y1, x, bar_y1 + tick_len], fill='black', width=outline_w)
        bbox = draw.textbbox((0, 0), label, font=font_tick)
        tw = bbox[2] - bbox[0]
        draw.text((x - tw / 2, bar_y1 + tick_len + 22), label, fill='black', font=font_tick)
    return out

def generate_metric(metric: str, source_files: dict[str, Path], unused=None) -> tuple[Path, Path]:
    raw_dir = OUT_ROOT / f'{metric}_Ropt_original_panel_original_base_map'
    panel_dir = OUT_ROOT / f'{metric}_Ropt_original_panel_without_title_annotations'
    raw_dir.mkdir(parents=True, exist_ok=True)
    panel_dir.mkdir(parents=True, exist_ok=True)
    for pathway, source_nc in source_files.items():
        for season_code, old_season_name, cell_index in SEASONS:
            compat_nc = make_compatible_nc(metric, pathway, source_nc, season_code)
            raw_panel = raw_dir / f'{metric}_{pathway}_{season_code}_Ropt_original_base_map_600dpi.jpg'
            final_panel = panel_dir / f'{metric}_{pathway}_{season_code}_Ropt_without_title_annotations_600dpi.jpg'
            draw_reference_panel(compat_nc, raw_panel)
            letter = 'abcd'[cell_index] if pathway == 'SPEI' else 'efgh'[cell_index]
            postprocess_panel(raw_panel, final_panel, letter, season_code)
            print(f'finished panel: {metric} {pathway} {season_code}')
    compact = stitch_no_stretch(panel_dir, metric)
    titled = add_column_titles(compact, metric)
    final = append_colorbar(titled)
    out_jpg = OUT_ROOT / f'{metric}_20years_Ropt_SPEI_SSMIrz_compact_row_spacing_with_colorbar_preserve_aspect_ratio_600dpi.jpg'
    out_pdf = OUT_ROOT / f'{metric}_20years_Ropt_SPEI_SSMIrz_compact_row_spacing_with_colorbar_preserve_aspect_ratio_600dpi.pdf'
    final.save(out_jpg, dpi=(600, 600), quality=95, subsampling=0)
    final.save(out_pdf, resolution=600)
    qa = {'metric': metric, 'source_files': {k: str(v) for k, v in source_files.items()}, 'raw_panel_dir': str(raw_dir), 'final_panel_dir': str(panel_dir), 'final_jpg': str(out_jpg), 'final_pdf': str(out_pdf), 'dpi': 600, 'panel_layout': 'same as final SIF figure', 'crop_top_px': CROP_TOP, 'row_gap_px': ROW_GAP, 'col_gap_px': COL_GAP, 'colorbar_outline_and_tick_width_px': COLORBAR_OUTLINE_W}
    qa_path = OUT_ROOT / f'{metric}_20years_Ropt_figuresQA.json'
    qa_path.write_text(json.dumps(qa, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'final {metric}: {out_jpg}')
    return (out_jpg, out_pdf)

def main() -> None:
    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    for metric, source_files in METRICS.items():
        generate_metric(metric, source_files)
if __name__ == '__main__':
    main()
